# Second Normal Form (2NF)

The previous notebook brought your Library into First Normal Form (1NF) by
splitting comma-separated authors and genres into their own junction tables.
1NF guarantees atomic values. The next step, Second Normal Form (2NF),
concerns tables with **composite candidate keys**, where a non-key attribute
might depend on only *part* of the key rather than the whole thing.

Your current Library tables all carry single-column keys (`book_id`,
`member_id`, `loan_id`, and so on), so a partial dependency cannot arise
and they are already in 2NF by construction. To study 2NF clearly we use a
small **constructed teaching example**: a university enrollment record with a
two-column key.

In [ ]:
from rel_db_learning import Relation
from rel_db_learning.widgets.self_assessment import fd_check, multiple_choice
import pandas as pd

## A university enrollment table

A university records which student is taking which course, along with the
official course name and the grade the student earned. Each row represents
one student enrolled in one course:

```
enrollment(student_id, course_id, course_name, grade)
```

A student can take many courses and a course can have many students so the
primary key must be the pair `(student_id, course_id)`. No single column
is enough on its own.

In [ ]:
data = {
    "student_id":  ["S1",        "S2",        "S3",        "S1",        "S2",        "S4"],
    "course_id":   ["CS101",     "CS101",     "CS101",     "MA201",     "MA201",     "PH301"],
    "course_name": ["Databases", "Databases", "Databases", "Calculus",  "Calculus",  "Physics"],
    "grade":       ["A",         "B",         "A",         "B",         "A",         "C"],
}
df = pd.DataFrame(data)
df

## Redundancy and anomalies

Look at the table above. The name "Databases" appears three times, once
for each student in CS101. "Calculus" appears twice for MA201. The
course name is a fact about the course alone, yet we are forced to repeat it
every time a student enrolls.

This redundancy causes the usual three anomalies:

- **Update anomaly:** renaming "Databases" to "Introduction to Databases"
  requires updating every row for CS101. Miss one and the data contradicts itself.
- **Insert anomaly:** you cannot record a new course until at least one student
  enrolls, because `student_id` is part of the key and cannot be null.
- **Delete anomaly:** remove the last student enrolled in PH301 and you lose
  the fact that PH301 is called "Physics".

## Partial dependencies and 2NF

Two terms to define before stating the rule.

A **prime attribute** is any attribute that appears in at least one candidate
key. A **non-prime attribute** is one that does not appear in any candidate key.

A **partial dependency** exists when a non-prime attribute is functionally
determined by a *proper subset* of a candidate key, that is, by part of the
key rather than the whole key.

**Second Normal Form (2NF):** a relation is in 2NF if and only if it is in 1NF
and no non-prime attribute is partially dependent on any candidate key.

In other words, every non-prime attribute must depend on the **full** key
not just part of it.

In [ ]:
enrollment = Relation(
    ["student_id", "course_id", "course_name", "grade"],
    name="enrollment",
)
enrollment.add_fd(["student_id", "course_id"], ["grade"])
enrollment.add_fd(["course_id"], ["course_name"])
print(enrollment)

In [ ]:
enrollment.candidate_keys()

In [ ]:
print("is_2nf:     ", enrollment.is_2nf())
print("normal_form:", enrollment.normal_form())

The candidate key is the composite set `{student_id, course_id}`.

The non-prime attributes are `course_name` and `grade` (neither appears in
any candidate key). Of these:

- `grade` depends on `{student_id, course_id}` as a whole. That is a full
  functional dependency, which is fine.
- `course_name` depends on `course_id` alone. Since `course_id` is only
  **part** of the composite key this is a **partial dependency**.

One partial dependency is enough to fail 2NF so the enrollment relation is
in 1NF only which matches `normal_form()` returning `"1NF"` above.

## The fix: decompose along the partial dependency

The partial dependency `course_id -> course_name` tells us that course name
is a fact **about a course**, not about an enrollment. We separate the two
concerns into two tables:

- `course(course_id, course_name)`: one row per course, course name stored exactly once.
- `enrollment(student_id, course_id, grade)`: one row per enrollment.

The two tables are joined on `course_id` whenever the course name is needed.
No fact is duplicated, and every non-prime attribute now depends on the
whole key of its table.

In [ ]:
course = Relation(["course_id", "course_name"], name="course")
course.add_fd("course_id", "course_name")

enrollment_clean = Relation(["student_id", "course_id", "grade"], name="enrollment_clean")
enrollment_clean.add_fd(["student_id", "course_id"], "grade")

print("course is_2nf:", course.is_2nf(),"  normal_form:", course.normal_form())
print("enrollment_clean is_2nf:", enrollment_clean.is_2nf(),"  normal_form:", enrollment_clean.normal_form())

Both decomposed tables satisfy 2NF (and in fact BCNF since each has a
single-column candidate key that determines everything else).

The course name now lives in exactly one row in `course`. Renaming a course
is a single update adding a new course does not require a student to be
enrolled first and removing the last enrollment in a course does not erase
the course name. Each fact lives in exactly one place.

## Back to the Library: why 2NF needed its own example

After the 1NF notebook your Library gained two junction tables: `book_author`
(`book_id`, `author_id`) and `book_genre` (`book_id`, `genre_id`). Both have
composite primary keys. Could they have a partial dependency?

A partial dependency requires a **non-prime attribute** one not in any
candidate key. In `book_author` and `book_genre` there are no columns beyond
the two key columns so there are no non-prime attributes at all. Nothing can
partially depend on part of the key because there is nothing to depend on it.
The tables are trivially in 2NF (and in BCNF) which the tool confirms below.

In [ ]:
book_author = Relation(["book_id", "author_id"], name="book_author")
# No functional dependencies beyond the key identifying itself.
print("book_author is_2nf:    ", book_author.is_2nf())
print("book_author normal_form:", book_author.normal_form())

## Self-assessment

In [ ]:
multiple_choice(
    question=(
        "A relation has a composite candidate key {X, Y} and a non-prime attribute Z. "
        "The FD X -&gt; Z holds. What kind of dependency is X -&gt; Z?"
    ),
    options=[
        "A full functional dependency on the candidate key",
        "A partial dependency on the candidate key",
        "A transitive dependency",
        "A trivial dependency",
    ],
    correct_index=1,
    explanation=(
        "X alone is only part of the composite key {X, Y}, so Z depending on X "
        "is a partial dependency. A full dependency would require both X and Y "
        "together to determine Z."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "In enrollment(student_id, course_id, course_name, grade) with candidate key "
        "{student_id, course_id}, why does the relation violate 2NF?"
    ),
    options=[
        "grade depends on student_id alone, which is only part of the key",
        "course_name depends on course_id alone, which is only part of the key",
        "course_name depends on the full key {student_id, course_id}",
        "The relation has no candidate key",
    ],
    correct_index=1,
    explanation=(
        "The FD course_id -&gt; course_name shows that course_name is determined by "
        "course_id alone. Since course_id is only part of the composite candidate key, "
        "this is a partial dependency and 2NF is violated."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "Which decomposition of enrollment(student_id, course_id, course_name, grade) "
        "removes the partial dependency?"
    ),
    options=[
        "course(course_id, course_name) and enrollment(student_id, course_id, grade)",
        "enrollment(student_id, course_name, grade) and course(course_id, student_id)",
        "enrollment(student_id, course_id, course_name) and grade(student_id, grade)",
        "No decomposition is needed; enrollment is already in 2NF",
    ],
    correct_index=0,
    explanation=(
        "Splitting into course(course_id, course_name) and enrollment(student_id, course_id, grade) "
        "gives each fact exactly one home. In course, course_name depends on the full key "
        "course_id. In enrollment, grade depends on the full composite key "
        "{student_id, course_id}. Neither table has a partial dependency."
    ),
)

### Exercise (a): candidate keys

Consider a supplier-part database that records which supplier supplies which
part, the quantity supplied and the city where the supplier is based:

```
supplier_part(supplier_id, part_id, supplier_city, quantity)
```

The functional dependencies are:
- `{supplier_id, part_id} -> quantity`
- `supplier_id -> supplier_city`

There is exactly one candidate key. It is composite: no single attribute is
enough to identify a row. Think about which two attributes together determine
all the others then fill in `my_keys` below.

In [ ]:
supplier_part = Relation(
    ["supplier_id", "part_id", "supplier_city", "quantity"],
    name="supplier_part",
)
supplier_part.add_fd(["supplier_id", "part_id"], "quantity")
supplier_part.add_fd("supplier_id", "supplier_city")
print(supplier_part)

In [ ]:
my_keys = [{"???", "???"}]  # Replace each ??? with an attribute name
fd_check(supplier_part, "candidate_keys", student_answer=my_keys)

### Exercise (b): is supplier_part in 2NF?

Before running the cell below, predict: is `supplier_part` in 2NF?
Think about the candidate key you found above and whether any non-prime
attribute depends on only part of it. Then run the cell to confirm.

In [ ]:
print("is_2nf:     ", supplier_part.is_2nf())
print("normal_form:", supplier_part.normal_form())

## What's next

Your Library tables including the new junction tables from the 1NF notebook,
are all in 2NF because none of them has a non-prime attribute that depends on
only part of a key.

The next notebook, 3NF, returns to the live Library. The `member` table still
stores `city` and `country` together, and `city -> country` is a transitive
dependency through a non-key column. The `book` table has a similar problem
with publisher details. Third Normal Form eliminates these.